# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

## Resolución

### Paso 0. Qué he visto en la web antes de programar

Mirando el código HTML de Books to Scrape (clic derecho → *Inspeccionar*):

**Página de listado** (`catalogue/page-1.html` a `page-50.html`, 20 libros por página):

| Dato | Dónde está en el HTML |
|---|---|
| cada libro | `<article class="product_pod">` |
| título completo | atributo `title` del enlace `<h3><a title="...">` (el texto visible viene cortado con "...") |
| enlace a la ficha | atributo `href` de ese mismo enlace (es relativo, hay que completarlo) |
| precio | `<p class="price_color">£51.77</p>` |
| valoración | la **clase** del párrafo: `<p class="star-rating Three">`. No hay número, hay una palabra |
| página siguiente | `<li class="next"><a href="page-2.html">` (no existe en la última página) |

**Ficha de cada libro:**

| Dato | Dónde está en el HTML |
|---|---|
| UPC y disponibilidad | tabla `<table class="table table-striped">`, filas con `<th>` (nombre) y `<td>` (valor) |
| género | la "miga de pan" (`<ul class="breadcrumb">`): Home › Books › **Género** › Título, es decir, el tercer elemento |
| descripción | el `<p>` que va justo después de `<div id="product_description">` (algunos libros no tienen) |

### Plan

1. Recorrer las 50 páginas del listado siguiendo el botón *next*.
2. En cada libro del listado, leer precio y valoración y **filtrar ahí mismo**.
3. Solo para los libros que pasan el filtro, abrir su ficha y sacar UPC, género, disponibilidad y descripción.

**Por qué filtrar en el listado y no después:** precio y valoración ya aparecen en el listado. Si abriera las 1000 fichas y filtrara al final, haría 1000 peticiones; así solo abro las de los libros que cumplen las condiciones. Es más rápido y carga menos la web.

### Paso 1. Librerías y constantes

- `requests` descarga el HTML de una página.
- `BeautifulSoup` lo convierte en un objeto en el que puedo buscar etiquetas con selectores CSS (`select`, `select_one`).
- `urljoin` completa los enlaces relativos: `urljoin(".../catalogue/page-1.html", "page-2.html")` → `.../catalogue/page-2.html`.
- `RATINGS` traduce la palabra de la clase (`"Three"`) a número (`3`).

In [ ]:
# Your solution goes here

import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/"
RATINGS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

### Paso 2. Descargar una página: `get_soup(url)`

**Qué hace:** descarga la página y la devuelve ya convertida en `BeautifulSoup`.

- `timeout=10`: si la web no responde en 10 segundos, da error en lugar de quedarse esperando para siempre.
- `raise_for_status()`: si la respuesta es un error (404, 500...), lo lanza en ese momento, en vez de intentar leer una página de error como si fuera un libro.
- `response.encoding = "utf-8"`: esta web no declara bien su codificación y `requests` la adivina mal; sin esta línea, el símbolo de la libra sale como `Â£`.

**Por qué una función:** se usa para las páginas del listado y para cada ficha. Así la lógica de descarga está en un solo sitio.

In [ ]:
def get_soup(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    response.encoding = "utf-8"
    return BeautifulSoup(response.text, "html.parser")

### Paso 3. Leer los libros de una página del listado: `parse_listing(soup, page_url)`

Para cada `article.product_pod` saco título, enlace, precio y valoración.

- **Precio:** el texto es `£51.77`. Con una expresión regular (`re.search(r"\d+\.\d+", ...)`) me quedo solo con el número y lo paso a `float`. **Por qué no `replace("£", "")`:** si la codificación fallara y apareciera `Â£`, el `replace` dejaría la `Â` y `float()` daría error; la expresión regular funciona en los dos casos.
- **Valoración:** `p["class"]` devuelve la lista de clases, `["star-rating", "Three"]`. La segunda es la palabra que traduzco con `RATINGS`.

In [ ]:
def parse_listing(soup, page_url):
    books = []
    for article in soup.select("article.product_pod"):
        link = article.select_one("h3 a")
        price_text = article.select_one("p.price_color").get_text()
        rating_word = article.select_one("p.star-rating")["class"][1]
        books.append({
            "title": link["title"],
            "url": urljoin(page_url, link["href"]),
            "price": float(re.search(r"\d+\.\d+", price_text).group()),
            "rating": RATINGS[rating_word],
        })
    return books

### Paso 4. Leer la ficha de un libro: `parse_book_page(soup)`

- **Tabla de información:** la convierto en un diccionario `{th: td}` con una comprensión de diccionario; así leo `info["UPC"]` y `info["Availability"]` por su nombre, sin depender del orden de las filas.
- **Género:** `ul.breadcrumb li` devuelve los cuatro elementos de la miga de pan; el género es el tercero (índice 2).
- **Descripción:** el selector `#product_description + p` significa "el `<p>` que va justo después del elemento con id `product_description`". Si el libro no tiene descripción, `select_one` devuelve `None` y guardo `None` en lugar de dar error.

In [ ]:
def parse_book_page(soup):
    info = {row.th.get_text(strip=True): row.td.get_text(strip=True)
            for row in soup.select("table.table-striped tr")}
    genre = soup.select("ul.breadcrumb li")[2].get_text(strip=True)
    description_tag = soup.select_one("#product_description + p")
    return {
        "UPC": info["UPC"],
        "Genre": genre,
        "Availability": info["Availability"],
        "Description": description_tag.get_text(strip=True) if description_tag else None,
    }

### Paso 5. La función principal: `scrape_books(min_rating, max_price)`

1. Empieza en la página 1 del catálogo.
2. Lee los libros de la página y se queda con los que tienen `rating >= min_rating` **y** `price <= max_price`.
3. Para cada uno, abre la ficha, junta los datos del listado y de la ficha, y guarda la fila.
4. Busca el enlace *next*. Si existe, pasa a la página siguiente; si no (última página), termina el bucle.
5. Devuelve un DataFrame con las columnas en el orden que pide el enunciado.

**Por qué seguir el enlace *next* y no hacer `range(1, 51)`:** si mañana la web tiene 49 o 60 páginas, el código sigue funcionando.

**`time.sleep(0.1)`:** una pausa corta entre peticiones. Es una norma de cortesía al hacer scraping, para no saturar el servidor.

**Tiempo:** con los valores del enunciado tarda alrededor de un minuto: 50 páginas de listado más 75 fichas, en lugar de las 1000 fichas que habría que abrir sin filtrar antes.

In [ ]:
def scrape_books(min_rating, max_price):
    rows = []
    page_url = urljoin(BASE_URL, "catalogue/page-1.html")

    while page_url:
        soup = get_soup(page_url)

        for book in parse_listing(soup, page_url):
            if book["rating"] >= min_rating and book["price"] <= max_price:
                details = parse_book_page(get_soup(book["url"]))
                rows.append({
                    "UPC": details["UPC"],
                    "Title": book["title"],
                    "Price (£)": book["price"],
                    "Rating": book["rating"],
                    "Genre": details["Genre"],
                    "Availability": details["Availability"],
                    "Description": details["Description"],
                })
                time.sleep(0.1)

        next_link = soup.select_one("li.next a")
        page_url = urljoin(page_url, next_link["href"]) if next_link else None
        time.sleep(0.1)

    columns = ["UPC", "Title", "Price (£)", "Rating", "Genre", "Availability", "Description"]
    return pd.DataFrame(rows, columns=columns)

### Paso 6. Ejecutar con los valores del enunciado: valoración de 4 o más y precio de hasta 20 £

In [ ]:
books = scrape_books(min_rating=4, max_price=20)
print(books.shape)
books.head()

### Paso 7. Comprobaciones

Verifico que el filtro se ha aplicado bien (ningún libro por debajo de 4 estrellas ni por encima de 20 £) y que no hay libros repetidos.

In [ ]:
print("Minimum rating:", books["Rating"].min())
print("Maximum price:", books["Price (£)"].max())
print("Repeated UPCs:", books["UPC"].duplicated().sum())
print("Books without description:", books["Description"].isna().sum())

### Paso 8. Un primer análisis para el equipo editorial

El objetivo del lab es entender qué géneros tienen libros bien valorados. Agrupo por género: cuántos libros hay y cuál es su precio medio.

In [ ]:
genre_summary = (
    books.groupby("Genre")
    .agg(books=("UPC", "count"), avg_price=("Price (£)", "mean"), avg_rating=("Rating", "mean"))
    .round(2)
    .sort_values("books", ascending=False)
)
genre_summary.head(10)

**Resultado esperado** (comprobado en la web el 30/09/2026): 75 libros con 4 o más estrellas y precio de hasta 20 £, de 1000 que tiene la tienda.

**Conclusiones:**

- El "género" más frecuente es **Default**, con 10 libros. No es un género real: es la categoría que la web pone a los libros sin clasificar. Para el análisis editorial conviene tratarlo aparte o como "sin género".
- Entre los géneros reales destacan **Sequential Art** (cómic y novela gráfica, 9 libros), **Young Adult** (7) y **Nonfiction** (6). Son los que concentran más libros bien valorados y baratos.
- Todos los libros seleccionados tienen descripción y están en stock, así que la columna `Availability` no discrimina en esta web; en una tienda real sería clave.

### Paso 9. Probar con otros valores

El enunciado pide comprobar que la función sirve para otras búsquedas. Por ejemplo, solo libros de 5 estrellas y hasta 15 £:

In [ ]:
top_cheap = scrape_books(min_rating=5, max_price=15)
print(top_cheap.shape)
top_cheap[["Title", "Price (£)", "Rating", "Genre"]].head()

Con 5 estrellas y hasta 15 £ salen 23 libros: la misma función sirve para cualquier combinación de valoración y precio sin tocar el código.